# Exact Knapsack (ILP) Team Formation — Using Uploaded Skill Datasets

This notebook implements an **exact** team selection solver using a **0–1 Integer Linear Program (ILP)**:

- Choose up to **K** researchers (team size constraint)
- Maximize **weighted coverage** of the proposal’s required skills
- Uses uploaded datasets:
  - `/mnt/data/large_researcher_skills.csv`
  - `/mnt/data/large_proposal_skills.csv`

It will:
1. Load & parse skills into Python sets  
2. Build **IDF** skill weights (rarity) from researchers  
3. Solve an **exact ILP** per proposal using `PuLP` (CBC solver)  
4. Output teams + coverage + goodness and optionally save results

> If `pulp` is missing, the notebook provides a small **exact brute-force fallback** for tiny candidate pools (useful for quick testing).


In [1]:

# === 1) SETUP ===
import pandas as pd
import numpy as np
import ast
import math
import random
from collections import Counter
import os
import datetime

RESEARCHER_SKILLS_PATH = "../data/input_data/Set_4/large_researcher_skills.csv"
PROPOSAL_SKILLS_PATH   = "../data/input_data/Set_4/large_proposal_skills.csv"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Paths:")
print(" -", RESEARCHER_SKILLS_PATH)
print(" -", PROPOSAL_SKILLS_PATH)


Paths:
 - ../data/input_data/Set_4/large_researcher_skills.csv
 - ../data/input_data/Set_4/large_proposal_skills.csv


In [2]:

# === 2) LOAD DATASETS ===
researchers_df = pd.read_csv(RESEARCHER_SKILLS_PATH)
proposals_df   = pd.read_csv(PROPOSAL_SKILLS_PATH)

print("Researchers:", researchers_df.shape)
print("Proposals:  ", proposals_df.shape)

display(researchers_df.head(3))
display(proposals_df.head(3))


Researchers: (2000, 3)
Proposals:   (500, 3)


,Unnamed: 0,researcher_name,skills
0,0,Carmen Meyer,"{'ceramics', 'artificial intelligence', 'neuro..."
1,1,Danielle Rodriguez PhD,"{'project management', 'graph theory', 'biolog..."
2,2,Tyler Hill DVM,"{'artificial intelligence', 'deep learning', '..."


,Unnamed: 0,nsf_proposal_links_v0,skills
0,0,https://www.nsf.gov/pubs/2026/nsf26000/nsf2600...,"{'python', 'optimization', 'deep learning', 's..."
1,1,https://www.nsf.gov/pubs/2026/nsf26001/nsf2600...,"{'data science', 'python', 'deep learning', 'n..."
2,2,https://www.nsf.gov/pubs/2026/nsf26002/nsf2600...,"{'structural health monitoring', 'machine lear..."


In [3]:

# === 3) PARSE SKILL SETS ===
def parse_skill_set(x):
    if pd.isna(x):
        return set()
    if isinstance(x, (set, list, tuple)):
        return set(x)
    try:
        val = ast.literal_eval(x)
        return set(val) if isinstance(val, (set, list, tuple)) else set()
    except Exception:
        s = str(x).strip()
        s = s.strip("{}")
        parts = [p.strip().strip("'").strip('"') for p in s.split(",") if p.strip()]
        return set(parts)

# Expected columns (based on uploaded files)
RESEARCHER_NAME_COL = "researcher_name"
PROPOSAL_LINK_COL   = "nsf_proposal_links_v0"
SKILLS_COL          = "skills"

researchers_df["skill_set"] = researchers_df[SKILLS_COL].apply(parse_skill_set)
proposals_df["skill_set"]   = proposals_df[SKILLS_COL].apply(parse_skill_set)

# Build maps
researcher_skills = dict(zip(researchers_df[RESEARCHER_NAME_COL], researchers_df["skill_set"]))
proposal_skills   = dict(zip(proposals_df[PROPOSAL_LINK_COL], proposals_df["skill_set"]))

all_researchers = list(researcher_skills.keys())
all_proposals   = list(proposal_skills.keys())

print("Parsed.")
print("Researchers in map:", len(all_researchers))
print("Proposals in map:  ", len(all_proposals))
print("Example researcher skill count:", len(researcher_skills[all_researchers[0]]))
print("Example proposal skill count:  ", len(proposal_skills[all_proposals[0]]))


Parsed.
Researchers in map: 1968
Proposals in map:   500
Example researcher skill count: 4
Example proposal skill count:   4


In [4]:
import random

def team_value_unweighted(req_skills, covered_skills):
    """Old definition: |covered| / |req|."""
    if not req_skills:
        return 0.0
    return len(covered_skills) / len(req_skills)

def build_team_old_cover_until_one(
    p_link,
    proposal_skills,
    researcher_skills,
    K=8,                 # optional cap; old method can run without, but good safety
    target_r=None,       # optional starting researcher
    allow_duplicates=False  # if False, never add same researcher twice
):
    req = proposal_skills.get(p_link, set())
    if not req:
        return [], {"coverage": 0.0, "covered": set(), "req": set()}

    all_researchers = list(researcher_skills.keys())

    team = []
    covered = set()

    # Seed with target researcher if provided
    if target_r is not None:
        team.append(target_r)
        covered |= (researcher_skills.get(target_r, set()) & req)

    # Keep adding until full coverage or can't improve or hit K
    while team_value_unweighted(req, covered) < 1.0 and len(team) < K:
        best_r = None
        best_gain = 0

        for r in all_researchers:
            if (not allow_duplicates) and (r in team):
                continue

            new_skills = (researcher_skills.get(r, set()) & req) - covered
            gain = len(new_skills)  # old unweighted gain

            if gain > best_gain:
                best_gain = gain
                best_r = r

        # Stop if nobody adds any new skills
        if best_r is None or best_gain == 0:
            break

        team.append(best_r)
        covered |= (researcher_skills.get(best_r, set()) & req)

    debug = {
        "req": req,
        "covered": covered,
        "coverage": team_value_unweighted(req, covered),
        "team_size": len(team),
    }
    return team, debug


In [6]:
import os, sys
import M1

# If this fails, uncomment and point to the folder containing M1.py
# sys.path.append("../path/to/your/code_folder")
# import M1


In [7]:
def ultra_goodness(p_link, team, proposal_skills, researcher_skills):
    # apply_ultra_metric(req_skills, team, pseudo_skills_map)
    return M1.apply_ultra_metric(
        proposal_skills.get(p_link, set()),
        team,
        researcher_skills
    )


In [8]:
def pick_best_target_for_proposal_unweighted(p_link, proposal_skills, researcher_skills):
    req = proposal_skills.get(p_link, set())
    if not req:
        return random.choice(list(researcher_skills.keys()))

    best_r, best_score = None, -1
    for r, skills in researcher_skills.items():
        score = len(skills & req)   # unweighted overlap count
        if score > best_score:
            best_score = score
            best_r = r
    return best_r if best_r is not None else random.choice(list(researcher_skills.keys()))


In [9]:
def run_old_algorithm_for_proposals(
    proposal_links,
    proposal_skills,
    researcher_skills,
    K=8,
    use_best_target=True
):
    rows = []

    for idx, p_link in enumerate(proposal_links):
        if use_best_target:
            target = pick_best_target_for_proposal_unweighted(p_link, proposal_skills, researcher_skills)
        else:
            target = None

        team, dbg = build_team_old_cover_until_one(
            p_link=p_link,
            proposal_skills=proposal_skills,
            researcher_skills=researcher_skills,
            K=K,
            target_r=target
        )

        req = dbg["req"]
        covered = dbg["covered"]

        goodness_ultra = ultra_goodness(p_link, team, proposal_skills, researcher_skills)

        rows.append({
            "proposal_link": p_link,
            "target_researcher": target,
            "team": team,
            "team_size": len(team),
            "req_skills": len(req),
            "covered_skills": len(covered),
            "coverage_unweighted": dbg["coverage"],     # your old value (0..1)
            "goodness_ultra": goodness_ultra            # your paper metric
        })

        if (idx + 1) % 25 == 0:
            print(f"Processed {idx+1}/{len(proposal_links)} proposals...")

    return pd.DataFrame(rows)


In [10]:
subset = all_proposals[:25]   # change to all_proposals for full run
results_df = run_old_algorithm_for_proposals(
    subset,
    proposal_skills,
    researcher_skills,
    K=8,
    use_best_target=True
)

display(results_df.head())


Processed 25/25 proposals...


,proposal_link,target_researcher,team,team_size,req_skills,covered_skills,coverage_unweighted,goodness_ultra
0,https://www.nsf.gov/pubs/2026/nsf26000/nsf2600...,Hayley Gutierrez,[Hayley Gutierrez],1,4,4,1.0,0.400
1,https://www.nsf.gov/pubs/2026/nsf26001/nsf2600...,Robert Smith,"[Robert Smith, Donna Stone]",2,6,6,1.0,0.425
2,https://www.nsf.gov/pubs/2026/nsf26002/nsf2600...,Marissa Flores,"[Marissa Flores, Heidi Johnson]",2,4,4,1.0,0.425
3,https://www.nsf.gov/pubs/2026/nsf26003/nsf2600...,Casey Webb,"[Casey Webb, Colin Graham]",2,4,4,1.0,0.425
4,https://www.nsf.gov/pubs/2026/nsf26004/nsf2600...,Elizabeth Keller,[Elizabeth Keller],1,4,4,1.0,0.400


In [11]:
OUT_DIR = "../data/output_data/old_algorithm_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"{OUT_DIR}/old_algo_results_{len(results_df)}_{timestamp}.csv"

results_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


Saved: ../data/output_data/old_algorithm_outputs/old_algo_results_25_20260204_220852.csv
